Step 2.1 — New notebook mein first cell

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

Step 2.2 — Dataset load karo

In [4]:
df = pd.read_csv("../data/raw/financial_fraud_detection_dataset.csv")

df.head()

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Amount,Merchant_Category,Payment_Method,Device_Type,Location,Is_International,Previous_Transactions,Average_Spend,Account_Age_Days,Suspicious_Keyword,Fraudulent
0,T100000,CUST3252,04-10-2023 07:45,37.54,Travel,PayPal,POS,Bengaluru,0,94,417.40,1492,No,0
1,T100001,CUST1630,26-09-2023 08:29,240.81,Utilities,PayPal,Mobile,Bengaluru,0,76,335.47,66,No,0
2,T100002,CUST7852,18-07-2023 15:54,105.34,Entertainment,Debit Card,Mobile,Bengaluru,0,60,432.03,216,No,0
3,T100003,CUST4892,16-10-2023 17:10,73.04,Utilities,NetBanking,Desktop,Kolkata,0,68,488.30,449,No,0
4,T100004,CUST8831,09-09-2023 00:46,13.57,Utilities,Credit Card,Desktop,Bengaluru,0,57,437.03,754,No,0


In [5]:
df.shape

(5000, 14)

Step 2.3 — Data types check

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Transaction_ID         5000 non-null   str    
 1   Customer_ID            5000 non-null   str    
 2   Transaction_Date       5000 non-null   str    
 3   Transaction_Amount     5000 non-null   float64
 4   Merchant_Category      5000 non-null   str    
 5   Payment_Method         5000 non-null   str    
 6   Device_Type            5000 non-null   str    
 7   Location               5000 non-null   str    
 8   Is_International       5000 non-null   int64  
 9   Previous_Transactions  5000 non-null   int64  
 10  Average_Spend          5000 non-null   float64
 11  Account_Age_Days       5000 non-null   int64  
 12  Suspicious_Keyword     5000 non-null   str    
 13  Fraudulent             5000 non-null   int64  
dtypes: float64(2), int64(4), str(8)
memory usage: 547.0 KB


Step 2.4 — Transaction Date convert

In [12]:
df["Transaction_Date"] = pd.to_datetime(
    df["Transaction_Date"],
    format="%d-%m-%Y %H:%M"
)

df["Transaction_Date"].dtype

dtype('<M8[us]')

In [13]:
df[["Transaction_Date"]].head()

,Transaction_Date
0,2023-10-04 07:45:00
1,2023-09-26 08:29:00
2,2023-07-18 15:54:00
3,2023-10-16 17:10:00
4,2023-09-09 00:46:00


In [14]:
print(df["Transaction_Date"].min())
print(df["Transaction_Date"].max())

2023-01-01 02:16:00
2024-02-21 15:34:00


Step 2.5 — Date se useful features

In [15]:
df["Transaction_Year"] = df["Transaction_Date"].dt.year
df["Transaction_Month"] = df["Transaction_Date"].dt.month
df["Transaction_Day"] = df["Transaction_Date"].dt.day
df["Transaction_DayOfWeek"] = df["Transaction_Date"].dt.dayofweek

In [16]:
df[
    [
        "Transaction_Date",
        "Transaction_Year",
        "Transaction_Month",
        "Transaction_Day",
        "Transaction_DayOfWeek"
    ]
].head()

,Transaction_Date,Transaction_Year,Transaction_Month,Transaction_Day,Transaction_DayOfWeek
0,2023-10-04 07:45:00,2023,10,4,2
1,2023-09-26 08:29:00,2023,9,26,1
2,2023-07-18 15:54:00,2023,7,18,1
3,2023-10-16 17:10:00,2023,10,16,0
4,2023-09-09 00:46:00,2023,9,9,5


Step 2.6 — Original date remove

In [17]:
df.drop("Transaction_Date", axis=1, inplace=True)

In [18]:
df.drop(["Transaction_ID", "Customer_ID"], axis=1, inplace=True)

Step 2.7 — Target separate karna

In [19]:
X = df.drop("Fraudulent", axis=1)
y = df["Fraudulent"]

In [20]:
X.shape

(5000, 14)

In [21]:
y.shape

(5000,)

In [22]:
y.value_counts()

Fraudulent
0    4518
1     482
Name: count, dtype: int64

Step 2.8 — Numerical vs categorical columns identify

In [23]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Transaction_Amount', 'Is_International', 'Previous_Transactions', 'Average_Spend', 'Account_Age_Days']

Categorical Features:
['Merchant_Category', 'Payment_Method', 'Device_Type', 'Location', 'Suspicious_Keyword']


C:\Users\vashi\AppData\Local\Temp\ipykernel_22292\4232826477.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


Step 2.9 — Feature values check

In [24]:
for col in categorical_features:
    print(f"\n{col}:")
    print(X[col].value_counts())


Merchant_Category:
Merchant_Category
Food             684
Electronics      650
Entertainment    630
Grocery          629
Fashion          608
Health           606
Travel           599
Utilities        594
Name: count, dtype: int64

Payment_Method:
Payment_Method
NetBanking     1037
PayPal         1022
Debit Card     1008
Credit Card     972
UPI             961
Name: count, dtype: int64

Device_Type:
Device_Type
Desktop    1679
POS        1673
Mobile     1648
Name: count, dtype: int64

Location:
Location
Delhi        758
Mumbai       741
Hyderabad    725
Pune         722
Kolkata      713
Bengaluru    686
Chennai      655
Name: count, dtype: int64

Suspicious_Keyword:
Suspicious_Keyword
No     4740
Yes     260
Name: count, dtype: int64


In [25]:
X[numerical_features].describe()

,Transaction_Amount,Is_International,Previous_Transactions,Average_Spend,Account_Age_Days
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,79.180664,0.088200,99.414000,258.377132,1011.158200
std,78.964045,0.283614,57.022968,140.230639,568.633834
min,0.000000,0.000000,1.000000,10.010000,30.000000
25%,22.365000,0.000000,50.000000,138.772500,530.000000
50%,55.455000,0.000000,100.000000,261.810000,1012.500000
75%,110.297500,0.000000,148.000000,378.525000,1496.000000
max,653.800000,1.000000,199.000000,499.990000,1999.000000


In [26]:
print("Is_International:")
print(X["Is_International"].value_counts())

print("\nSuspicious_Keyword:")
print(X["Suspicious_Keyword"].value_counts())

Is_International:
Is_International
0    4559
1     441
Name: count, dtype: int64

Suspicious_Keyword:
Suspicious_Keyword
No     4740
Yes     260
Name: count, dtype: int64


Step 2.10 — Train/Test Split

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [28]:
print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (4000, 14)
Testing set : (1000, 14)

Training target distribution:
Fraudulent
0    0.9035
1    0.0965
Name: proportion, dtype: float64

Testing target distribution:
Fraudulent
0    0.904
1    0.096
Name: proportion, dtype: float64


Step 2.11 — Preprocessing Pipeline

In [29]:
numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

Step 2.12 — ColumnTransformer

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Step 2.13 — Fit ONLY on training data

In [37]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training data processed successfully!")
print("Testing data processed successfully!")

Training data processed successfully!
Testing data processed successfully!


Step 2.14 — Shape check

In [38]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (4000, 14)
Processed training shape: (4000, 30)
Original testing shape: (1000, 14)
Processed testing shape: (1000, 30)


Step 2.15 — Feature names check

In [39]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

print("\nFeature names:")
for feature in feature_names:
    print(feature)

Total processed features: 30

Feature names:
num__Transaction_Amount
num__Is_International
num__Previous_Transactions
num__Average_Spend
num__Account_Age_Days
cat__Merchant_Category_Electronics
cat__Merchant_Category_Entertainment
cat__Merchant_Category_Fashion
cat__Merchant_Category_Food
cat__Merchant_Category_Grocery
cat__Merchant_Category_Health
cat__Merchant_Category_Travel
cat__Merchant_Category_Utilities
cat__Payment_Method_Credit Card
cat__Payment_Method_Debit Card
cat__Payment_Method_NetBanking
cat__Payment_Method_PayPal
cat__Payment_Method_UPI
cat__Device_Type_Desktop
cat__Device_Type_Mobile
cat__Device_Type_POS
cat__Location_Bengaluru
cat__Location_Chennai
cat__Location_Delhi
cat__Location_Hyderabad
cat__Location_Kolkata
cat__Location_Mumbai
cat__Location_Pune
cat__Suspicious_Keyword_No
cat__Suspicious_Keyword_Yes


Step 2.16 — Convert processed data into DataFrame

In [40]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [41]:
X_train_processed_df.head()

,num__Transaction_Amount,num__Is_International,num__Previous_Transactions,num__Average_Spend,num__Account_Age_Days,cat__Merchant_Category_Electronics,cat__Merchant_Category_Entertainment,cat__Merchant_Category_Fashion,cat__Merchant_Category_Food,cat__Merchant_Category_Grocery,...,cat__Device_Type_POS,cat__Location_Bengaluru,cat__Location_Chennai,cat__Location_Delhi,cat__Location_Hyderabad,cat__Location_Kolkata,cat__Location_Mumbai,cat__Location_Pune,cat__Suspicious_Keyword_No,cat__Suspicious_Keyword_Yes
1876,-0.101150,-0.308205,-1.038190,-1.179653,1.150629,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4538,-0.920980,-0.308205,0.591587,0.793598,-0.994671,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4063,-0.711167,-0.308205,-1.423729,-1.415321,0.139829,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2920,-0.607848,3.244592,-1.336106,-1.500941,1.332644,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
937,1.487357,-0.308205,1.169895,1.111248,0.123925,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [42]:
X_train_processed_df.shape

(4000, 30)

Step 2.17 — Scaling verify karo

In [43]:
X_train_processed_df[
    [col for col in feature_names if col.startswith("num__")]
].describe().T

,count,mean,std,min,25%,50%,75%,max
num__Transaction_Amount,4000.0,-7.283063e-17,1.000125,-1.007418,-0.721099,-0.303220,0.405135,7.291152
num__Is_International,4000.0,7.993606e-18,1.000125,-0.308205,-0.308205,-0.308205,-0.308205,3.244592
num__Previous_Transactions,4000.0,1.953993e-17,1.000125,-1.721645,-0.867326,-0.004245,0.854455,1.748204
num__Average_Spend,4000.0,-1.421085e-17,1.000125,-1.773426,-0.857526,0.035613,0.857849,1.722573
num__Account_Age_Days,4000.0,9.636736e-17,1.000125,-1.729798,-0.846232,0.004643,0.844915,1.749688


Step 2.18 — Final target verification

In [44]:
print("y_train:")
print(y_train.value_counts())

print("\ny_test:")
print(y_test.value_counts())

y_train:
Fraudulent
0    3614
1     386
Name: count, dtype: int64

y_test:
Fraudulent
0    904
1     96
Name: count, dtype: int64


Step 2.19 — Processed data save karna

In [45]:
X_train_processed_df.to_csv(
    "../data/processed/X_train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "../data/processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

Step 2.20 — Final verification

In [46]:
import os

processed_files = [
    "X_train_processed.csv",
    "X_test_processed.csv",
    "y_train.csv",
    "y_test.csv"
]

for file in processed_files:
    path = os.path.join("../data/processed", file)
    print(file, "→", "Created" if os.path.exists(path) else "Missing")

X_train_processed.csv → Created
X_test_processed.csv → Created
y_train.csv → Created
y_test.csv → Created
